In [20]:
import pandas as pd
import numpy as np
import seaborn as sns
import json 
import os
import datetime
from datetime import datetime  

In [21]:
path = f'src/data/raw/debt_to_penny_{datetime.today().strftime("%Y%m%d")}/treasury_raw.json'

with open(path) as f:
    raw = json.load(f)

df = pd.DataFrame(raw['data'])

In [22]:
df.dtypes

print(df.columns.tolist())

['record_date', 'debt_held_public_amt', 'intragov_hold_amt', 'tot_pub_debt_out_amt', 'src_line_nbr', 'record_fiscal_year', 'record_fiscal_quarter', 'record_calendar_year', 'record_calendar_quarter', 'record_calendar_month', 'record_calendar_day']


In [23]:
df.head()

,record_date,debt_held_public_amt,intragov_hold_amt,tot_pub_debt_out_amt,src_line_nbr,record_fiscal_year,record_fiscal_quarter,record_calendar_year,record_calendar_quarter,record_calendar_month,record_calendar_day
0,2026-05-13,31269241615132.64,7673382066715.04,38942623681847.68,1,2026,3,2026,2,05,13
1,2026-05-12,31268302404373.30,7699992655432.05,38968295059805.35,1,2026,3,2026,2,05,12
2,2026-05-11,31262108319021.06,7684692242388.08,38946800561409.14,1,2026,3,2026,2,05,11
3,2026-05-08,31260345834577.53,7677129508008.27,38937475342585.80,1,2026,3,2026,2,05,08
4,2026-05-07,31262325604447.63,7669326114354.46,38931651718802.09,1,2026,3,2026,2,05,07


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8307 entries, 0 to 8306
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   record_date              8307 non-null   object
 1   debt_held_public_amt     8307 non-null   object
 2   intragov_hold_amt        8307 non-null   object
 3   tot_pub_debt_out_amt     8307 non-null   object
 4   src_line_nbr             8307 non-null   object
 5   record_fiscal_year       8307 non-null   object
 6   record_fiscal_quarter    8307 non-null   object
 7   record_calendar_year     8307 non-null   object
 8   record_calendar_quarter  8307 non-null   object
 9   record_calendar_month    8307 non-null   object
 10  record_calendar_day      8307 non-null   object
dtypes: object(11)
memory usage: 714.0+ KB


# Cleaning:

In [25]:
# Treasury "null" string literal → real NaN

df = df.replace('null', np.nan)

In [26]:
# Numeric coercion on all amount columns
amt_cols = df.filter(like='amt').columns.tolist()
df[amt_cols] = df[amt_cols].apply(pd.to_numeric, errors='coerce')

In [27]:
# Date column
df['record_date'] = pd.to_datetime(df['record_date']).dt.date

In [28]:
# Nullable Int64 for calendar parts (check which exist in your frame first)
calendar_cols = ['record_calendar_year', 'record_calendar_month', 
                  'record_calendar_quarter', 'record_calendar_day',
                  'record_fiscal_year', 'record_fiscal_quarter']
for col in calendar_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

In [29]:
# Sanity checks
print(df.dtypes)
print(df.isna().sum())
print(df.shape)

record_date                 object
debt_held_public_amt       float64
intragov_hold_amt          float64
tot_pub_debt_out_amt       float64
src_line_nbr                object
record_fiscal_year           Int64
record_fiscal_quarter        Int64
record_calendar_year         Int64
record_calendar_quarter      Int64
record_calendar_month        Int64
record_calendar_day          Int64
dtype: object
record_date                   0
debt_held_public_amt       2958
intragov_hold_amt          2958
tot_pub_debt_out_amt          0
src_line_nbr                  0
record_fiscal_year            0
record_fiscal_quarter         0
record_calendar_year          0
record_calendar_quarter       0
record_calendar_month         0
record_calendar_day           0
dtype: int64
(8307, 11)


# Write to CSV

In [30]:
os.makedirs('src/data/curated', exist_ok=True)
df.to_csv('src/data/curated/treasury_debt_clean.csv', index=False)

print(f"Wrote {len(df)} rows to data/curated/treasury_debt_clean.csv")

Wrote 8307 rows to data/curated/treasury_debt_clean.csv


In [31]:
print(df.dtypes)
print(df.head(3))

record_date                 object
debt_held_public_amt       float64
intragov_hold_amt          float64
tot_pub_debt_out_amt       float64
src_line_nbr                object
record_fiscal_year           Int64
record_fiscal_quarter        Int64
record_calendar_year         Int64
record_calendar_quarter      Int64
record_calendar_month        Int64
record_calendar_day          Int64
dtype: object
  record_date  debt_held_public_amt  intragov_hold_amt  tot_pub_debt_out_amt  \
0  2026-05-13          3.126924e+13       7.673382e+12          3.894262e+13   
1  2026-05-12          3.126830e+13       7.699993e+12          3.896830e+13   
2  2026-05-11          3.126211e+13       7.684692e+12          3.894680e+13   

  src_line_nbr  record_fiscal_year  record_fiscal_quarter  \
0            1                2026                      3   
1            1                2026                      3   
2            1                2026                      3   

   record_calendar_year  record_cal